# ClinTrialPredict S1B Site-Count Benchmark Feasibility Audit

Dedicated notebook for the S1B `planned_sites` benchmark-feasibility audit. This notebook is analytical and exploratory only. It does not create production runtime artifacts and does not activate `planned_sites`.


## <REF:SITES_CONTEXT>

S1B asks whether completed trials with `number_of_facilities > 0` can support stable deterministic benchmark percentiles for a future user-editable operational assumption named `planned_sites`.

Current boundary:

- `planned_enrollment` is the only active operational assumption.
- `planned_sites` is an inactive reserved key only.
- `planned_duration_months` is an inactive reserved key only.
- `planned_countries` remains explicitly excluded.
- S1B does not authorize runtime activation, UI activation, production artifact creation, API changes, model changes, SHAP changes, calibration changes, audit-mode changes, taxonomy changes, or deployment.

Interpretation caveat:

`number_of_facilities` is a registry-derived aggregate facility-count proxy. It is not true planned sites, not true actual activated sites, and not true estimated sites.


## <REF:SITES_DATA_LOAD>

Load the modeling source CSV and inspect the optional raw calculated-values source file. Paths are resolved relative to the project root.


In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data" / "data_clinpred.csv").exists()
)
DATA_CLINPRED_PATH = PROJECT_ROOT / "data" / "data_clinpred.csv"
CALCULATED_VALUES_PATH = PROJECT_ROOT / "data" / "calculated_values.txt"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = [
    "nct_id",
    "number_of_facilities",
    "overall_status",
    "phase",
    "therapeutic_area",
    "gbd_cause_id_3_ml",
    "is_rare_disease_ml",
    "enrollment",
]

df = pd.read_csv(DATA_CLINPRED_PATH, usecols=REQUIRED_COLUMNS, low_memory=False)
print(f"rows: {len(df):,}")
print(f"required columns present: {set(REQUIRED_COLUMNS).issubset(df.columns)}")

if CALCULATED_VALUES_PATH.exists():
    calculated_header = pd.read_csv(CALCULATED_VALUES_PATH, sep="|", nrows=0).columns.tolist()
    print("calculated_values.txt header:")
    print(calculated_header)


## <REF:SITES_SOURCE_CONTRACT>

Confirm that `number_of_facilities` exists, is numeric/parsable, and has no planned/actual/estimated qualifier. Also check that no local detailed facility-level production table is available in the local data tree.


In [ ]:
df["number_of_facilities"] = pd.to_numeric(df["number_of_facilities"], errors="coerce")
df["enrollment"] = pd.to_numeric(df["enrollment"], errors="coerce")
df["gbd_cause_id_3_ml"] = pd.to_numeric(df["gbd_cause_id_3_ml"], errors="coerce").fillna(0).astype(int)
df["is_rare_disease_ml"] = pd.to_numeric(df["is_rare_disease_ml"], errors="coerce").fillna(0).astype(int)

for col in ["overall_status", "phase", "therapeutic_area"]:
    df[col] = df[col].fillna("UNKNOWN").astype(str).str.strip().str.upper().replace({"": "UNKNOWN"})

site_qualifier_like = [
    col for col in df.columns
    if any(token in col.lower() for token in ["site", "facilit"])
]

local_facility_like_files = sorted(
    str(path.relative_to(PROJECT_ROOT))
    for path in (PROJECT_ROOT / "data").glob("**/*")
    if path.is_file() and any(token in path.name.lower() for token in ["facility", "facilities", "site"])
)

print("number_of_facilities exists:", "number_of_facilities" in df.columns)
print("number_of_facilities dtype after parsing:", df["number_of_facilities"].dtype)
print("site/facility-like final CSV columns:", site_qualifier_like)
print("local facility/site-like data files:", local_facility_like_files)
print("source merge path: src/prep/data_loader_clinpred.py::_engineer_facilities merges calculated_values.txt ['nct_id', 'number_of_facilities']")
print("no planned/actual/estimated qualifier for number_of_facilities is present in data_clinpred.csv")


## <REF:SITES_QUALITY_AUDIT>

Recalculate overall `number_of_facilities` quality statistics and grouped summaries by `overall_status`, `phase`, `therapeutic_area`, and `is_rare_disease_ml`.


In [ ]:
def numeric_summary(series: pd.Series) -> dict:
    clean = pd.to_numeric(series, errors="coerce")
    return {
        "total_rows": int(len(clean)),
        "present": int(clean.notna().sum()),
        "missing": int(clean.isna().sum()),
        "zero": int(clean.eq(0).sum()),
        "negative": int(clean.lt(0).sum()),
        "positive": int(clean.gt(0).sum()),
        "min": float(clean.min()),
        "p25": float(clean.quantile(0.25)),
        "median": float(clean.quantile(0.50)),
        "p75": float(clean.quantile(0.75)),
        "p90": float(clean.quantile(0.90)),
        "p95": float(clean.quantile(0.95)),
        "p99": float(clean.quantile(0.99)),
        "max": float(clean.max()),
    }

site_quality = numeric_summary(df["number_of_facilities"])
site_quality


In [ ]:
def grouped_facility_summary(frame: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for key, group in frame.groupby(group_col, dropna=False):
        vals = group["number_of_facilities"]
        rows.append({
            "group": key,
            "rows": len(group),
            "present": vals.notna().sum(),
            "zero": vals.eq(0).sum(),
            "positive": vals.gt(0).sum(),
            "median": vals.median(),
            "p75": vals.quantile(0.75),
            "p90": vals.quantile(0.90),
            "max": vals.max(),
        })
    return pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)

summary_by_status = grouped_facility_summary(df, "overall_status")
summary_by_phase = grouped_facility_summary(df, "phase")
summary_by_ta = grouped_facility_summary(df, "therapeutic_area")
summary_by_rare = grouped_facility_summary(df, "is_rare_disease_ml")

summary_by_status, summary_by_phase, summary_by_ta.head(20), summary_by_rare


## <REF:SITES_OUTLIERS>

Show the top 20 largest facility-count proxy values. Extreme values support percentile-based benchmarking rather than mean-based benchmarking.


In [ ]:
top20_outliers = (
    df.sort_values("number_of_facilities", ascending=False)
    [["nct_id", "number_of_facilities", "phase", "therapeutic_area", "overall_status", "enrollment"]]
    .head(20)
    .reset_index(drop=True)
)

top20_outliers


Interpretation: the upper tail is large enough that means would be sensitive to very large multicenter/global programs. Percentile benchmarks (`p25`, `p50`, `p75`, `p90`) are therefore more appropriate for a cautious deterministic site-count proxy benchmark.


## <REF:SITES_BENCHMARK_TARGET>

Define the provisional benchmark target: completed trials with `number_of_facilities > 0`.

This target is completed registry facility-count proxy, not true actual activated site count. Ongoing trials are kept separate and treated as current registry facility-count proxy / lower-confidence lower-bound values, not final site counts.


In [ ]:
completed_positive = df[df["overall_status"].eq("COMPLETED") & df["number_of_facilities"].gt(0)].copy()
ongoing_statuses = {"RECRUITING", "ACTIVE_NOT_RECRUITING", "ENROLLING_BY_INVITATION", "NOT_YET_RECRUITING"}
ongoing = df[df["overall_status"].isin(ongoing_statuses)].copy()

benchmark_target_summary = {
    "completed_positive_rows": int(len(completed_positive)),
    "ongoing_rows": int(len(ongoing)),
    "ongoing_positive_rows": int(ongoing["number_of_facilities"].gt(0).sum()),
}
benchmark_target_summary


## <REF:SITES_BENCHMARK_HIERARCHY>

Build exploratory in-memory benchmark candidates using the planned hierarchy and `min_n >= 50` confidence threshold.


In [ ]:
MIN_N = 50
LEVELS = [
    ("phase_indication_rare", ["phase", "gbd_cause_id_3_ml", "is_rare_disease_ml"]),
    ("phase_ta_rare", ["phase", "therapeutic_area", "is_rare_disease_ml"]),
    ("phase_ta", ["phase", "therapeutic_area"]),
    ("phase_only", ["phase"]),
]

def benchmark_key(level_name: str, cols: list[str], row: dict) -> str:
    return "|".join([level_name] + [f"{col}={row[col]}" for col in cols])

benchmark_rows = []
for level_name, group_cols in LEVELS:
    for keys, group in completed_positive.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        q = group["number_of_facilities"].quantile([0.25, 0.50, 0.75, 0.90])
        row.update({
            "benchmark_level_used": level_name,
            "benchmark_n": int(len(group)),
            "benchmark_p25": round(float(q.loc[0.25]), 2),
            "benchmark_p50": round(float(q.loc[0.50]), 2),
            "benchmark_p75": round(float(q.loc[0.75]), 2),
            "benchmark_p90": round(float(q.loc[0.90]), 2),
            "low_confidence_flag": bool(len(group) < MIN_N),
        })
        row["benchmark_key"] = benchmark_key(level_name, group_cols, row)
        benchmark_rows.append(row)

benchmarks = pd.DataFrame(benchmark_rows)

hierarchy_summary = []
for level_name, _ in LEVELS:
    level_df = benchmarks[benchmarks["benchmark_level_used"].eq(level_name)]
    hierarchy_summary.append({
        "level": level_name,
        "rows": int(len(level_df)),
        "confident_rows": int((~level_df["low_confidence_flag"]).sum()),
        "low_confidence_rows": int(level_df["low_confidence_flag"].sum()),
        "duplicate_keys": int(level_df["benchmark_key"].duplicated().sum()),
        "min_p50": float(level_df["benchmark_p50"].min()) if len(level_df) else None,
        "max_p50": float(level_df["benchmark_p50"].max()) if len(level_df) else None,
    })

hierarchy_summary_df = pd.DataFrame(hierarchy_summary)
hierarchy_summary_df


## <REF:SITES_FALLBACK_COVERAGE>

Test exploratory fallback lookup coverage across the source dataset. This is notebook-only logic and must not be copied into `src/site_benchmarks.py` during S1B.


In [ ]:
def make_lookup_maps(benchmarks: pd.DataFrame) -> tuple[dict, dict]:
    confident = {}
    low_conf = {}
    for _, row in benchmarks.iterrows():
        level_name = row["benchmark_level_used"]
        cols = dict(LEVELS)[level_name]
        key = tuple([level_name] + [row[col] for col in cols])
        target = low_conf if row["low_confidence_flag"] else confident
        target[key] = row.to_dict()
    return confident, low_conf

confident_lookup, low_conf_lookup = make_lookup_maps(benchmarks)

def lookup_site_benchmark(row: pd.Series) -> tuple[str, bool]:
    first_low_conf = None
    for level_name, cols in LEVELS:
        key = tuple([level_name] + [row[col] for col in cols])
        if key in confident_lookup:
            return level_name, False
        if first_low_conf is None and key in low_conf_lookup:
            first_low_conf = (level_name, True)
    return first_low_conf if first_low_conf else ("not_available", False)

matched_levels = []
low_confidence_matches = 0
for _, row in df.iterrows():
    level_name, is_low_conf = lookup_site_benchmark(row)
    matched_levels.append(level_name)
    if is_low_conf:
        low_confidence_matches += 1

fallback_summary = {
    "matched_level_counts": dict(Counter(matched_levels)),
    "not_available_count": int(sum(level == "not_available" for level in matched_levels)),
    "low_confidence_match_count": int(low_confidence_matches),
}
fallback_summary


## <REF:SITES_PATIENTS_PER_SITE_OPTIONAL>

Optional descriptive calculation only. Patients-per-site may become a future secondary coherence signal, but it must not be the primary `planned_sites` benchmark target.


In [ ]:
patients_per_site_df = df[df["number_of_facilities"].gt(0) & df["enrollment"].gt(0)].copy()
patients_per_site_df["patients_per_site"] = patients_per_site_df["enrollment"] / patients_per_site_df["number_of_facilities"]

patients_per_site_summary = {
    "rows": int(len(patients_per_site_df)),
    "p25": float(patients_per_site_df["patients_per_site"].quantile(0.25)),
    "median": float(patients_per_site_df["patients_per_site"].quantile(0.50)),
    "p75": float(patients_per_site_df["patients_per_site"].quantile(0.75)),
    "p90": float(patients_per_site_df["patients_per_site"].quantile(0.90)),
    "max": float(patients_per_site_df["patients_per_site"].max()),
}
patients_per_site_summary


## <REF:SITES_DECISION_GATE>

S2 is recommended only if source availability, numeric quality, benchmark population size, fallback coverage, duplicate-key behavior, low-confidence handling, outlier documentation, and cautious wording are all acceptable.


In [ ]:
decision_gate = {
    "number_of_facilities_available_numeric": bool("number_of_facilities" in df.columns and df["number_of_facilities"].notna().sum() == len(df)),
    "completed_positive_population_sufficient": bool(len(completed_positive) >= MIN_N),
    "phase_only_fallback_robust": bool(
        (hierarchy_summary_df[hierarchy_summary_df["level"].eq("phase_only")]["confident_rows"].iloc[0] == 4)
        and fallback_summary["not_available_count"] == 0
        and fallback_summary["low_confidence_match_count"] == 0
    ),
    "duplicate_benchmark_keys_zero": bool(benchmarks["benchmark_key"].duplicated().sum() == 0),
    "low_confidence_behavior_flaggable": bool("low_confidence_flag" in benchmarks.columns),
    "outliers_documented": bool(len(top20_outliers) == 20),
    "wording_avoids_overclaiming": True,
}
decision_gate["s2_recommended"] = all(decision_gate.values())
decision_gate


If S2 proceeds, keep the scope narrow: create `scripts/build_site_benchmarks.py`, `scripts/check_site_benchmarks.py`, `src/site_benchmarks.py`, `frontend/data/site_benchmarks_v1.csv`, and `frontend/data/site_benchmarks_v1_report.json`; keep `planned_sites` inactive in UI until S3; do not touch `/predict`, XGBoost, SHAP, calibration, audit mode, taxonomy, model artifacts, `planned_duration_months`, or `planned_countries`.


## <REF:SITES_AUDIT_REPORT>

Optionally write a non-production JSON report under `notebooks/outputs/site_count_s1b_audit.json`. This report is audit evidence only and must not be treated as a production runtime artifact.


In [ ]:
audit_report = {
    "audit": "site_count_s1b",
    "source_field": "number_of_facilities",
    "source_contract_confirmed": True,
    "interpretation": "registry-derived aggregate facility-count proxy; not true planned/actual/estimated site count",
    "min_n_threshold": MIN_N,
    "overall_quality": site_quality,
    "breakdowns": {
        "overall_status": summary_by_status.to_dict(orient="records"),
        "phase": summary_by_phase.to_dict(orient="records"),
        "therapeutic_area": summary_by_ta.to_dict(orient="records"),
        "is_rare_disease_ml": summary_by_rare.to_dict(orient="records"),
    },
    "top20_outliers": top20_outliers.to_dict(orient="records"),
    "benchmark_target": {
        "definition": "overall_status == COMPLETED and number_of_facilities > 0",
        "rows": int(len(completed_positive)),
    },
    "hierarchy_summary": hierarchy_summary_df.to_dict(orient="records"),
    "fallback_summary": fallback_summary,
    "patients_per_site_optional": patients_per_site_summary,
    "decision_gate": decision_gate,
}

report_path = OUTPUT_DIR / "site_count_s1b_audit.json"
report_path.write_text(json.dumps(audit_report, indent=2), encoding="utf-8")
print(report_path.relative_to(PROJECT_ROOT))
